# IMC Prosperity 4 — Round 1 Market Analysis (V2)

Restructured based on top-10 team approaches from Prosperity 3.

**Products:**
- **ASH_COATED_OSMIUM** — described as "volatile" with a "hidden pattern", but data shows within-day CV~0.0005, mean anchored at ~10,000 (actually STABLE)
- **INTARIAN_PEPPER_ROOT** — described as "steady" but drifts ~1000/day upward, within-day CV~0.025 (actually DRIFTING)

**Round 1 data notes:**
- buyer/seller columns are EMPTY (no counterparty IDs)
- Book depth: level 2 ~65% populated, level 3 ~2%
- ~4% of ticks have empty book (filtered out)

**Sections:**
1. Product Classification (within-day CV + drift detection)
2. Fair Value Estimation (4 methods incl. MM-filtered mid)
3. Spread & Edge Calibration
4. Mean Reversion vs Trend
5. Signal Predictiveness (multi-horizon)
6. Trade Pattern & Bot Analysis
7. Intraday Pattern Detection (hidden pattern search)
8. Spike Detection & Event Study
9. Cross-Product Analysis
10. Position & Inventory Risk
11. Parameter Sensitivity Grid
12. Strategy Decision Dashboard

In [ ]:
%matplotlib inline
import glob, os, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "figure.facecolor": "white",
    "axes.facecolor": "#f8f8f8", "axes.grid": True, "grid.alpha": 0.3,
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "legend.fontsize": 8, "figure.titlesize": 12,
})
PLOT_DIR = "plots"
os.makedirs(PLOT_DIR, exist_ok=True)
DAY_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
              "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

POSITION_LIMITS = {"ASH_COATED_OSMIUM": 80, "INTARIAN_PEPPER_ROOT": 80}
DEFAULT_LIMIT = 80
MM_SIZE_THRESHOLD = 10  # Orders >= this size are likely market-maker quotes

def savefig(fig, name):
    fig.savefig(os.path.join(PLOT_DIR, f"{name}.png"), bbox_inches="tight")

## Helper Functions

In [ ]:
def compute_features(df):
    d = df.copy()
    d["simple_mid"] = (d["bid_price_1"] + d["ask_price_1"]) / 2
    bv1 = d["bid_volume_1"].fillna(0); av1 = d["ask_volume_1"].fillna(0)
    denom = bv1 + av1
    d["microprice"] = np.where(denom > 0,
        (d["bid_price_1"] * av1 + d["ask_price_1"] * bv1) / denom, d["simple_mid"])

    bid_prices = d[["bid_price_1", "bid_price_2", "bid_price_3"]].values
    bid_vols   = d[["bid_volume_1", "bid_volume_2", "bid_volume_3"]].fillna(0).values
    ask_prices = d[["ask_price_1", "ask_price_2", "ask_price_3"]].values
    ask_vols   = d[["ask_volume_1", "ask_volume_2", "ask_volume_3"]].fillna(0).values

    bid_wall = np.array([bp[np.nanargmax(bv)] if np.nanmax(bv) > 0 else bp[0]
                         for bp, bv in zip(bid_prices, bid_vols)])
    ask_wall = np.array([ap[np.nanargmax(av)] if np.nanmax(av) > 0 else ap[0]
                         for ap, av in zip(ask_prices, ask_vols)])
    d["bid_wall_price"] = bid_wall; d["ask_wall_price"] = ask_wall
    d["wall_mid"] = (bid_wall + ask_wall) / 2

    # MM-filtered mid: only large orders (>= threshold) count as market-maker
    mm_bid = np.full(len(d), np.nan); mm_ask = np.full(len(d), np.nan)
    for i in range(len(d)):
        for lvl in range(3):
            if bid_vols[i, lvl] >= MM_SIZE_THRESHOLD:
                mm_bid[i] = bid_prices[i, lvl]; break
        for lvl in range(3):
            if ask_vols[i, lvl] >= MM_SIZE_THRESHOLD:
                mm_ask[i] = ask_prices[i, lvl]; break
    valid_mm = ~np.isnan(mm_bid) & ~np.isnan(mm_ask)
    d["mm_mid"] = np.where(valid_mm, (mm_bid + mm_ask) / 2, d["wall_mid"])

    d["spread"] = d["ask_price_1"] - d["bid_price_1"]
    total_bid = d[["bid_volume_1", "bid_volume_2", "bid_volume_3"]].fillna(0).sum(axis=1)
    total_ask = d[["ask_volume_1", "ask_volume_2", "ask_volume_3"]].fillna(0).sum(axis=1)
    d["imbalance"] = (total_bid - total_ask) / (total_bid + total_ask).replace(0, np.nan)

    d["ret"] = d["mid_price"].diff()
    d["rolling_vol"] = d["ret"].rolling(10).std()
    d["rolling_vol_20"] = d["ret"].rolling(20).std()
    rm20 = d["mid_price"].rolling(20).mean()
    rs20 = d["mid_price"].rolling(20).std()
    d["z20"] = (d["mid_price"] - rm20) / rs20.replace(0, np.nan)
    d["spike"] = d["ret"].abs() > 3 * d["rolling_vol"]

    d["ret1"] = d["ret"]
    d["micro_delta"] = d["microprice"] - d["simple_mid"]
    d["wall_delta"] = d["wall_mid"] - d["simple_mid"]
    d["mm_delta"] = d["mm_mid"] - d["simple_mid"]
    d["spread_chg"] = d["spread"].diff()
    for h in [1, 2, 5, 10, 20]:
        d[f"fwd_ret_{h}"] = d["mid_price"].diff(h).shift(-h)
    return d

## Section 0 — Data Loading

In [ ]:
price_files = sorted(glob.glob("prices_round_*_day_*.csv"))
trade_files = sorted(glob.glob("trades_round_*_day_*.csv"))

prices = pd.concat([pd.read_csv(f, sep=";") for f in price_files], ignore_index=True)
trades = pd.concat([pd.read_csv(f, sep=";") for f in trade_files], ignore_index=True)

for c in [c for c in prices.columns if c != "product"]:
    prices[c] = pd.to_numeric(prices[c], errors="coerce")
for c in ["price", "quantity", "timestamp"]:
    if c in trades.columns:
        trades[c] = pd.to_numeric(trades[c], errors="coerce")
if "symbol" in trades.columns:
    trades = trades.rename(columns={"symbol": "product"})

# Filter empty book rows
prices = prices[(prices["mid_price"] > 0) & prices["bid_price_1"].notna() & prices["ask_price_1"].notna()].reset_index(drop=True)

PRODUCTS = sorted(prices["product"].unique())
product_dfs = {p: prices[prices["product"] == p].copy().reset_index(drop=True) for p in PRODUCTS}
trade_dfs = {p: trades[trades["product"] == p].copy().reset_index(drop=True)
             for p in PRODUCTS if p in trades["product"].unique()}
FEAT = {p: compute_features(product_dfs[p]) for p in PRODUCTS}

print(f"Products: {PRODUCTS}")
print(f"Days: {sorted(prices['day'].unique())}")
print(f"Price rows: {len(prices):,} (after filtering), Trade rows: {len(trades):,}")
for p in PRODUCTS:
    df = FEAT[p]
    print(f"  {p:30s}  rows={len(df):>6,}  mid=[{df['mid_price'].min():.1f}, {df['mid_price'].max():.1f}]")

## Overview — Quick Look at the Data
Simple plots to get an intuitive feel for each product before diving into detailed analysis.


In [ ]:
# Overview: Summary statistics per product per day
for p in PRODUCTS:
    df = FEAT[p]
    print(f"\n{'='*60}")
    print(f"  {p}")
    print(f"{'='*60}")
    print(f"  Overall: {len(df):,} ticks, price range [{df['mid_price'].min():.0f}, {df['mid_price'].max():.0f}]")
    print(f"  Mean={df['mid_price'].mean():.1f}, Std={df['mid_price'].std():.1f}, "
          f"Median={df['mid_price'].median():.1f}")
    print(f"  Spread: mean={df['spread'].mean():.1f}, median={df['spread'].median():.1f}")
    print()
    days = sorted(df["day"].unique())
    print(f"  {'Day':>5s} | {'Ticks':>6s} | {'Open':>8s} | {'Close':>8s} | {'Low':>8s} | {'High':>8s} | {'Mean':>8s} | {'Std':>6s}")
    print(f"  {'-'*5}-+-{'-'*6}-+-{'-'*8}-+-{'-'*8}-+-{'-'*8}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
    for day in days:
        dd = df[df["day"] == day]["mid_price"]
        print(f"  {day:>5d} | {len(dd):>6d} | {dd.iloc[0]:>8.0f} | {dd.iloc[-1]:>8.0f} | "
              f"{dd.min():>8.0f} | {dd.max():>8.0f} | {dd.mean():>8.0f} | {dd.std():>6.1f}")
    if p in trade_dfs:
        tdf = trade_dfs[p]
        print(f"\n  Trades: {len(tdf):,} total, avg qty={tdf['quantity'].mean():.1f}, "
              f"price range [{tdf['price'].min():.0f}, {tdf['price'].max():.0f}]")


In [ ]:
# Overview: Raw price time series — one subplot per product
fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(14, 5*len(PRODUCTS)), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[idx, 0]
    df = FEAT[p]
    for di, day in enumerate(sorted(df["day"].unique())):
        dd = df[df["day"] == day]
        ax.plot(dd["timestamp"], dd["mid_price"], color=DAY_COLORS[di % len(DAY_COLORS)],
                alpha=0.8, linewidth=0.7, label=f"Day {day}")
    ax.set_title(f"{p} — Raw Mid Price", fontsize=11)
    ax.set_xlabel("Timestamp"); ax.set_ylabel("Price")
    ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "0A_overview_price")
plt.show()


In [ ]:
# Overview: Bid/Ask/Mid overlay — see the spread visually
fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(14, 5*len(PRODUCTS)), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[idx, 0]
    df = FEAT[p]
    step = max(1, len(df) // 3000)  # downsample for plotting
    ds = df.iloc[::step]
    ax.fill_between(ds["timestamp"], ds["bid_price_1"], ds["ask_price_1"],
                     alpha=0.3, color="skyblue", label="Bid-Ask Spread")
    ax.plot(ds["timestamp"], ds["mid_price"], color="black", linewidth=0.6,
            alpha=0.8, label="Mid Price")
    ax.set_title(f"{p} — Bid/Ask Envelope", fontsize=11)
    ax.set_xlabel("Timestamp"); ax.set_ylabel("Price")
    ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "0B_overview_bidask")
plt.show()


In [ ]:
# Overview: Return distribution — is it symmetric? fat tails?
fig, axes = plt.subplots(1, len(PRODUCTS), figsize=(7*len(PRODUCTS), 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0, idx]
    df = FEAT[p]
    rets = df["ret"].dropna()
    ax.hist(rets, bins=80, color="#5599dd", edgecolor="white", alpha=0.8, density=True)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.axvline(rets.mean(), color="red", linestyle="--", linewidth=1, label=f"Mean={rets.mean():.2f}")
    ax.set_title(f"{p}\nReturns: std={rets.std():.2f}, skew={rets.skew():.2f}, kurt={rets.kurtosis():.1f}", fontsize=9)
    ax.set_xlabel("Tick-to-tick Return"); ax.set_ylabel("Density")
    ax.legend(fontsize=7)
fig.suptitle("Overview — Return Distribution", fontsize=12, y=1.02)
fig.tight_layout()
savefig(fig, "0C_overview_returns")
plt.show()


In [ ]:
# Overview: Volume profile — where does liquidity sit?
fig, axes = plt.subplots(1, len(PRODUCTS), figsize=(7*len(PRODUCTS), 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0, idx]
    df = FEAT[p]
    # Aggregate volume at each price level (bid side + ask side)
    all_prices = []; all_vols = []
    for lvl in range(1, 4):
        bp = df[f"bid_price_{lvl}"].dropna(); bv = df[f"bid_volume_{lvl}"].dropna()
        ap = df[f"ask_price_{lvl}"].dropna(); av = df[f"ask_volume_{lvl}"].dropna()
        if len(bp) > 0 and len(bv) > 0:
            all_prices.extend(bp.values[:len(bv)]); all_vols.extend(bv.values[:len(bp)])
        if len(ap) > 0 and len(av) > 0:
            all_prices.extend(ap.values[:len(av)]); all_vols.extend(av.values[:len(ap)])
    vp = pd.DataFrame({"price": all_prices, "volume": all_vols})
    vp_agg = vp.groupby(vp["price"].round(0))["volume"].sum().sort_index()
    ax.barh(vp_agg.index, vp_agg.values, height=0.8, color="#5599dd", alpha=0.7)
    ax.set_title(f"{p} — Volume Profile (all levels)", fontsize=9)
    ax.set_ylabel("Price"); ax.set_xlabel("Total Volume")
fig.suptitle("Overview — Where Liquidity Concentrates", fontsize=12, y=1.02)
fig.tight_layout()
savefig(fig, "0D_overview_volume_profile")
plt.show()


In [ ]:
# Overview: Trade activity — when and how much?
fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(14, 5*len(PRODUCTS)), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[idx, 0]
    if p not in trade_dfs or len(trade_dfs[p]) == 0:
        ax.text(0.5, 0.5, "No trade data", transform=ax.transAxes, ha="center", fontsize=14)
        continue
    tdf = trade_dfs[p]
    scatter = ax.scatter(tdf["timestamp"], tdf["price"], s=tdf["quantity"]*3,
                         alpha=0.4, c=tdf["quantity"], cmap="YlOrRd", edgecolors="none")
    plt.colorbar(scatter, ax=ax, label="Quantity", shrink=0.8)
    ax.set_title(f"{p} — Trades (size = quantity)", fontsize=11)
    ax.set_xlabel("Timestamp"); ax.set_ylabel("Trade Price")
fig.tight_layout()
savefig(fig, "0E_overview_trades")
plt.show()


In [ ]:
# Overview: Daily price box plots — see range, median, outliers at a glance
fig, axes = plt.subplots(1, len(PRODUCTS), figsize=(7*len(PRODUCTS), 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0, idx]
    df = FEAT[p]
    days = sorted(df["day"].unique())
    data = [df[df["day"]==d]["mid_price"].values for d in days]
    bp = ax.boxplot(data, labels=[f"Day {d}" for d in days], patch_artist=True)
    for j, patch in enumerate(bp["boxes"]):
        patch.set_facecolor(DAY_COLORS[j % len(DAY_COLORS)])
        patch.set_alpha(0.6)
    ax.set_title(f"{p} — Daily Price Distribution", fontsize=11)
    ax.set_ylabel("Mid Price")
fig.suptitle("Overview — Daily Box Plots", fontsize=12, y=1.02)
fig.tight_layout()
savefig(fig, "0F_overview_boxplots")
plt.show()


---
# Section 1 — Product Classification
Uses **within-day CV** (not across-day, which is inflated by drift).
Also detects day-over-day drift — critical for INTARIAN_PEPPER_ROOT which shifts ~1000/day.

In [ ]:
def classify_product_within_day(df):
    days = sorted(df["day"].unique())
    day_cvs, day_means = [], []
    for day in days:
        dd = df[df["day"] == day]["mid_price"].dropna()
        if len(dd) > 10:
            day_cvs.append(dd.std() / dd.mean())
            day_means.append(dd.mean())
    avg_cv = np.mean(day_cvs) if day_cvs else 0
    drift_per_day = abs(np.mean(np.diff(day_means))) if len(day_means) >= 2 else 0
    drift_pct = drift_per_day / np.mean(day_means) if day_means else 0
    has_drift = drift_pct > 0.01
    stability = "STABLE" if avg_cv < 0.001 else ("DRIFTING" if avg_cv < 0.03 else "VOLATILE")
    if has_drift: stability = "DRIFTING"  # Override: strong drift always = DRIFTING
    return stability, avg_cv, drift_per_day, has_drift

# 1A: Price History with per-day means
n = len(PRODUCTS)
fig, axes = plt.subplots(1, n, figsize=(7*n, 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0, idx]; df = FEAT[p]
    for di, day in enumerate(sorted(df["day"].unique())):
        dd = df[df["day"] == day]
        ax.plot(dd["timestamp"], dd["mid_price"], color=DAY_COLORS[di%len(DAY_COLORS)],
                alpha=0.7, linewidth=0.6, label=f"Day {day}")
        ax.axhline(dd["mid_price"].mean(), color=DAY_COLORS[di%len(DAY_COLORS)],
                   linestyle=":", linewidth=0.8, alpha=0.5)
    ax.set_title(f"{p}\nDo daily means shift? (drift detection)", fontsize=9)
    ax.set_xlabel("Timestamp"); ax.set_ylabel("Mid Price"); ax.legend(fontsize=7)
fig.suptitle("1A — Price History (dotted = daily means)", fontsize=12, y=1.02)
fig.tight_layout(); savefig(fig, "1A_price_history"); plt.show()

In [ ]:
# 1B: Per-day distribution
VERDICTS = {}
fig, axes = plt.subplots(1, n, figsize=(7*n, 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0, idx]; df = FEAT[p]
    v, avg_cv, drift, has_drift = classify_product_within_day(df)
    VERDICTS[p] = v
    for di, day in enumerate(sorted(df["day"].unique())):
        mid = df[df["day"]==day]["mid_price"].dropna()
        ax.hist(mid, bins=40, color=DAY_COLORS[di%len(DAY_COLORS)], alpha=0.5, edgecolor="white", label=f"Day {day}")
    drift_str = f", drift={drift:.0f}/day" if has_drift else ""
    ax.set_title(f"{p} — CV={avg_cv:.6f}{drift_str} → {v}", fontsize=9)
    ax.set_xlabel("Mid Price"); ax.set_ylabel("Freq"); ax.legend(fontsize=7)
fig.suptitle("1B — Per-Day Distribution", fontsize=12, y=1.02); fig.tight_layout()
savefig(fig, "1B_price_distribution"); plt.show()

for p in PRODUCTS:
    v, cv, drift, hd = classify_product_within_day(FEAT[p])
    ds = f"  drift={drift:.0f}/day" if hd else ""
    print(f"  {p:30s}  within-day CV={cv:.6f}{ds}  → {v}")

In [ ]:
# 1C: Rolling Volatility
fig, axes = plt.subplots(1, n, figsize=(7*n, 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0, idx]; df = FEAT[p]
    for di, day in enumerate(sorted(df["day"].unique())):
        dd = df[df["day"]==day]
        ax.plot(dd["timestamp"], dd["rolling_vol_20"], color=DAY_COLORS[di%len(DAY_COLORS)],
                alpha=0.7, linewidth=0.6, label=f"Day {day}")
    ax.axhline(df["rolling_vol_20"].mean(), color="red", linestyle="--", linewidth=1.2,
               label=f"Mean={df['rolling_vol_20'].mean():.3f}")
    ax.set_title(f"{p} — Constant or regime-switching?", fontsize=9)
    ax.set_xlabel("Timestamp"); ax.set_ylabel("Rolling 20-tick Std"); ax.legend(fontsize=7)
fig.suptitle("1C — Volatility Regimes", fontsize=12, y=1.02); fig.tight_layout()
savefig(fig, "1C_rolling_volatility"); plt.show()

In [ ]:
# 1D: Day-over-day trend
fig, axes = plt.subplots(1, n, figsize=(7*n, 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0, idx]; df = FEAT[p]
    days = sorted(df["day"].unique())
    dm = [df[df["day"]==d]["mid_price"].mean() for d in days]
    ds = [df[df["day"]==d]["mid_price"].std() for d in days]
    ax.errorbar(days, dm, yerr=ds, fmt="o-", capsize=5, color="#1f77b4", linewidth=2, markersize=8)
    if len(days) >= 2:
        slope = np.polyfit(days, dm, 1)[0]
        ax.annotate(f"Slope = {slope:.1f}/day", xy=(0.05, 0.90), xycoords="axes fraction",
                    fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="orange"))
    ax.set_title(f"{p} — Day-over-day trend", fontsize=9)
    ax.set_xlabel("Day"); ax.set_ylabel("Mean ± 1σ")
fig.suptitle("1D — Daily Trend", fontsize=12, y=1.02); fig.tight_layout()
savefig(fig, "1D_daily_trend"); plt.show()

---
# Section 2 — Fair Value Estimation
Compares 4 estimators: simple mid, microprice, wall mid, MM-filtered mid (orders ≥ 10 lots).

For ASH_COATED_OSMIUM (STABLE): lowest-noise estimator → fixed anchor.
For INTARIAN_PEPPER_ROOT (DRIFTING): need dynamic tracking, but lowest-noise still best for instantaneous FV.

In [ ]:
# 2A: Fair Value Comparison
FV_RESULTS = {}
for p in PRODUCTS:
    df = FEAT[p]
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    axes[0].plot(df["timestamp"], df["simple_mid"], alpha=0.6, linewidth=0.5, label="Simple Mid", color="#1f77b4")
    axes[0].plot(df["timestamp"], df["microprice"], alpha=0.6, linewidth=0.5, label="Microprice", color="#ff7f0e")
    axes[0].plot(df["timestamp"], df["wall_mid"], alpha=0.6, linewidth=0.5, label="Wall Mid", color="#2ca02c")
    axes[0].plot(df["timestamp"], df["mm_mid"], alpha=0.6, linewidth=0.5, label=f"MM-Filt (≥{MM_SIZE_THRESHOLD})", color="#d62728")
    axes[0].set_ylabel("Price"); axes[0].legend(fontsize=7)
    axes[0].set_title(f"{p} — Which is most stable?", fontsize=9)
    axes[1].plot(df["timestamp"], df["microprice"]-df["simple_mid"], alpha=0.7, linewidth=0.5, color="#ff7f0e", label="Micro−Mid")
    axes[1].plot(df["timestamp"], df["mm_mid"]-df["simple_mid"], alpha=0.7, linewidth=0.5, color="#d62728", label="MM−Mid")
    axes[1].axhline(0, color="black", linewidth=0.5); axes[1].set_ylabel("Deviation"); axes[1].legend(fontsize=7)
    axes[2].plot(df["timestamp"], df["wall_mid"]-df["simple_mid"], alpha=0.7, linewidth=0.5, color="#2ca02c")
    axes[2].axhline(0, color="black", linewidth=0.5); axes[2].set_ylabel("Wall−Mid"); axes[2].set_xlabel("Timestamp")
    fig.suptitle(f"2A — {p}: Fair Value Comparison", fontsize=12, y=1.01); fig.tight_layout()
    savefig(fig, f"2A_fair_value_{p}"); plt.show()
    FV_RESULTS[p] = {k: df[k.replace("_std","")].std() for k in ["simple_mid_std","microprice_std","wall_mid_std","mm_mid_std"]}

In [ ]:
# 2B: Wall Quote Scatter
for p in PRODUCTS:
    df = FEAT[p]; fig, ax = plt.subplots(figsize=(14, 6))
    step = max(1, len(df)//4000); ds = df.iloc[::step]
    vb = ["bid_volume_1","bid_volume_2","bid_volume_3"]; va = ["ask_volume_1","ask_volume_2","ask_volume_3"]
    pb = ["bid_price_1","bid_price_2","bid_price_3"]; pa = ["ask_price_1","ask_price_2","ask_price_3"]
    allv = np.concatenate([ds[vb].fillna(0).values.flatten(), ds[va].fillna(0).values.flatten()])
    v75 = np.percentile(allv[allv>0], 75) if (allv>0).any() else 1
    for i in range(3):
        bvol=ds[vb[i]].fillna(0); bp=ds[pb[i]]; m=bvol>0
        ax.scatter(ds["timestamp"][m], bp[m], s=np.clip(bvol[m]/v75*15,2,60), alpha=0.15, color="blue", edgecolors="none")
        wm=m&(bvol>=MM_SIZE_THRESHOLD)
        if wm.any(): ax.scatter(ds["timestamp"][wm], bp[wm], s=np.clip(bvol[wm]/v75*30,20,120), alpha=0.5, color="blue", edgecolors="darkblue", linewidth=0.5, label="MM Bid" if i==0 else "")
        avol=ds[va[i]].fillna(0); ap=ds[pa[i]]; ma=avol>0
        ax.scatter(ds["timestamp"][ma], ap[ma], s=np.clip(avol[ma]/v75*15,2,60), alpha=0.15, color="red", edgecolors="none")
        wma=ma&(avol>=MM_SIZE_THRESHOLD)
        if wma.any(): ax.scatter(ds["timestamp"][wma], ap[wma], s=np.clip(avol[wma]/v75*30,20,120), alpha=0.5, color="red", edgecolors="darkred", linewidth=0.5, label="MM Ask" if i==0 else "")
    ax.plot(ds["timestamp"], ds["wall_mid"], color="black", linewidth=1, alpha=0.8, label="Wall Mid")
    ax.set_title(f"{p} — Book structure (large dots = MM ≥{MM_SIZE_THRESHOLD})", fontsize=9)
    ax.set_xlabel("Timestamp"); ax.set_ylabel("Price"); ax.legend(fontsize=7); fig.tight_layout()
    savefig(fig, f"2B_wall_quotes_{p}"); plt.show()

In [ ]:
# 2C: Stability Ranking
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(PRODUCTS)); w = 0.2
for i,(m,c,l) in enumerate(zip(["simple_mid_std","microprice_std","wall_mid_std","mm_mid_std"],
    ["#1f77b4","#ff7f0e","#2ca02c","#d62728"], ["Simple","Micro","Wall",f"MM(≥{MM_SIZE_THRESHOLD})"])):
    ax.bar(x+i*w, [FV_RESULTS[p][m] for p in PRODUCTS], w, label=l, color=c, alpha=0.8)
ax.set_xticks(x+1.5*w); ax.set_xticklabels(PRODUCTS)
ax.set_ylabel("Std of FV"); ax.set_title("2C — Lowest = best anchor"); ax.legend()
fig.tight_layout(); savefig(fig, "2C_fair_value_ranking"); plt.show()
for p in PRODUCTS:
    r = FV_RESULTS[p]; best = min(r, key=r.get).replace("_std","")
    print(f"  {p:30s}  simple={r['simple_mid_std']:.4f}  micro={r['microprice_std']:.4f}  wall={r['wall_mid_std']:.4f}  mm={r['mm_mid_std']:.4f}  → {best}")

---
# Section 3 — Spread & Edge Calibration
Determines take_edge and passive_edge parameters from spread distribution.

In [ ]:
SPREAD_RESULTS = {}
# 3A
fig, axes = plt.subplots(1, n, figsize=(7*n, 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0,idx]; df = FEAT[p]
    for di, day in enumerate(sorted(df["day"].unique())):
        dd = df[df["day"]==day]
        ax.plot(dd["timestamp"], dd["spread"], color=DAY_COLORS[di%len(DAY_COLORS)], alpha=0.6, linewidth=0.5, label=f"Day {day}")
    ax.axhline(df["spread"].dropna().mean(), color="black", linestyle="--", linewidth=1, label=f"Mean={df['spread'].mean():.2f}")
    ax.set_title(f"{p}", fontsize=9); ax.set_xlabel("Timestamp"); ax.set_ylabel("Spread"); ax.legend(fontsize=6)
fig.suptitle("3A — Spread Over Time", fontsize=12, y=1.02); fig.tight_layout(); savefig(fig, "3A_spread_time"); plt.show()

In [ ]:
# 3B
fig, axes = plt.subplots(1, n, figsize=(7*n, 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0,idx]; sp = FEAT[p]["spread"].dropna()
    ax.hist(sp, bins=60, color="#5599dd", edgecolor="white", alpha=0.8)
    p25,p50,p75 = sp.quantile(0.25), sp.quantile(0.5), sp.quantile(0.75)
    for pv,lb,cl in [(p25,"25th","green"),(p50,"50th","orange"),(p75,"75th","red")]:
        ax.axvline(pv, color=cl, linestyle="--", linewidth=1.2, label=f"{lb}={pv:.1f}")
    te=p25/4; pe=p50/3
    ax.annotate(f"take_edge ≈ {te:.2f}\npassive_edge ≈ {pe:.2f}", xy=(0.55,0.80), xycoords="axes fraction",
                fontsize=8, bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="orange"))
    ax.set_title(f"{p}", fontsize=9); ax.set_xlabel("Spread"); ax.set_ylabel("Freq"); ax.legend(fontsize=7)
    SPREAD_RESULTS[p] = {"mean":sp.mean(),"std":sp.std(),"p25":p25,"p50":p50,"p75":p75,"take_edge":te,"passive_edge":pe}
fig.suptitle("3B — Spread Distribution", fontsize=12, y=1.02); fig.tight_layout(); savefig(fig, "3B_spread_distribution"); plt.show()
for p in PRODUCTS:
    sr=SPREAD_RESULTS[p]; print(f"  {p:30s}  mean={sr['mean']:.2f}  p25/p50/p75={sr['p25']:.1f}/{sr['p50']:.1f}/{sr['p75']:.1f}  TE={sr['take_edge']:.2f}  PE={sr['passive_edge']:.2f}")

---
# Section 4 — Mean Reversion vs Trend
Negative lag-1 autocorrelation = mean reverting (market make aggressively).

In [ ]:
ACF_RESULTS = {}
lags = [1,2,3,5,10]
fig, axes = plt.subplots(1, n, figsize=(7*n, 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0,idx]; rets = FEAT[p]["ret"].dropna()
    acfs = [rets.autocorr(lag=l) for l in lags]
    ax.bar(range(len(lags)), acfs, color=["#d62728" if a<0 else "#1f77b4" for a in acfs], edgecolor="white", alpha=0.8)
    ax.set_xticks(range(len(lags))); ax.set_xticklabels([str(l) for l in lags])
    ax.axhline(0, color="black", linewidth=0.5); ax.axhline(0.05, color="gray", linestyle=":", linewidth=0.8); ax.axhline(-0.05, color="gray", linestyle=":", linewidth=0.8)
    ax.set_title(f"{p}  RED=mean revert  BLUE=trend", fontsize=8); ax.set_xlabel("Lag"); ax.set_ylabel("ACF")
    ACF_RESULTS[p] = {"lag1": acfs[0], "acfs": dict(zip(lags, acfs))}
fig.suptitle("4A — Return Autocorrelation", fontsize=12, y=1.02); fig.tight_layout(); savefig(fig, "4A_autocorrelation"); plt.show()
for p in PRODUCTS:
    l1=ACF_RESULTS[p]["lag1"]; v="MEAN REVERTING" if l1<-0.05 else ("TRENDING" if l1>0.05 else "NEUTRAL")
    print(f"  {p:30s}  lag-1={l1:.4f}  → {v}")

In [ ]:
# 4B: Z-Score
for p in PRODUCTS:
    df = FEAT[p]; fig, (ax1,ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    rm = df["mid_price"].rolling(20).mean(); rs = df["mid_price"].rolling(20).std()
    ax1.plot(df["timestamp"], df["mid_price"], alpha=0.7, linewidth=0.5, color="#1f77b4", label="Mid")
    ax1.plot(df["timestamp"], rm, color="black", linewidth=1, label="Roll 20 Mean")
    ax1.fill_between(df["timestamp"], df["mid_price"], rm, where=df["mid_price"]>rm+rs, alpha=0.3, color="red", label="Sell zone")
    ax1.fill_between(df["timestamp"], df["mid_price"], rm, where=df["mid_price"]<rm-rs, alpha=0.3, color="green", label="Buy zone")
    ax1.set_ylabel("Mid"); ax1.legend(fontsize=7)
    ax2.plot(df["timestamp"], df["z20"], alpha=0.7, linewidth=0.5, color="#ff7f0e")
    for lv in [1,-1,2,-2]: ax2.axhline(lv, color="gray", linestyle="--" if abs(lv)==1 else ":", linewidth=0.7)
    ax2.axhline(0, color="black", linewidth=0.5); ax2.set_ylabel("Z-Score"); ax2.set_xlabel("Timestamp")
    fig.suptitle(f"4B — {p}: Mean Reversion Zones", fontsize=10, y=1.01); fig.tight_layout()
    savefig(fig, f"4B_zscore_{p}"); plt.show()

---
# Section 5 — Signal Predictiveness (Multi-Horizon)
Tests which signals predict future returns at horizons 1, 2, 5, 10, 20 ticks.
Includes MM-delta — the deviation of MM-filtered mid from simple mid.

In [ ]:
SIGNAL_NAMES = ["ret1","z20","micro_delta","wall_delta","mm_delta","imbalance","spread_chg"]
HORIZONS = [1,2,5,10,20]
SIGNAL_RESULTS = {}

for p in PRODUCTS:
    df = FEAT[p]; results = {}
    for s in SIGNAL_NAMES:
        hc = {}
        for h in HORIZONS:
            valid = df[[s, f"fwd_ret_{h}"]].dropna()
            c = valid[s].corr(valid[f"fwd_ret_{h}"]) if len(valid)>30 else 0
            hc[h] = c if not np.isnan(c) else 0
        v1 = df[[s,"fwd_ret_1"]].dropna()
        hit = ((np.sign(v1[s])==np.sign(v1["fwd_ret_1"])) & (v1[s]!=0) & (v1["fwd_ret_1"]!=0)).mean() if len(v1)>30 else 0.5
        results[s] = {"corrs":hc, "hit":hit, "corr":hc.get(1,0)}
    SIGNAL_RESULTS[p] = results

    fig, ax = plt.subplots(figsize=(10, 5))
    ss = sorted(SIGNAL_NAMES, key=lambda s: abs(results[s]["corr"]))
    cs = [results[s]["corr"] for s in ss]
    ax.barh(range(len(ss)), cs, color=["#d62728" if c<0 else "#1f77b4" for c in cs], edgecolor="white", alpha=0.8)
    ax.set_yticks(range(len(ss))); ax.set_yticklabels(ss); ax.axvline(0, color="black", linewidth=0.5)
    ax.set_xlabel("Corr with next-tick return"); ax.set_title(f"{p} — Signal strength (h=1)", fontsize=9)
    fig.tight_layout(); savefig(fig, f"5A_signal_corr_{p}"); plt.show()

In [ ]:
# 5B: Signal Decay
for p in PRODUCTS:
    fig, ax = plt.subplots(figsize=(12, 6)); res = SIGNAL_RESULTS[p]
    for s in SIGNAL_NAMES:
        ax.plot(HORIZONS, [abs(res[s]["corrs"].get(h,0)) for h in HORIZONS], marker="o", linewidth=1.5, markersize=4, label=s, alpha=0.8)
    ax.set_xlabel("Horizon (ticks)"); ax.set_ylabel("|Correlation|")
    ax.set_title(f"{p} — Signal decay: which persist?", fontsize=9); ax.legend(fontsize=7, ncol=2); ax.set_xticks(HORIZONS)
    fig.tight_layout(); savefig(fig, f"5B_signal_decay_{p}"); plt.show()

In [ ]:
# 5C: Hit Rate
fig, axes = plt.subplots(1, len(PRODUCTS), figsize=(7*len(PRODUCTS), 5), squeeze=False)
for idx, p in enumerate(PRODUCTS):
    ax = axes[0,idx]; res = SIGNAL_RESULTS[p]
    hits = [res[s]["hit"] for s in SIGNAL_NAMES]
    ax.bar(range(len(SIGNAL_NAMES)), hits, color=["#2ca02c" if h>0.5 else "#d62728" for h in hits], edgecolor="white", alpha=0.8)
    ax.set_xticks(range(len(SIGNAL_NAMES))); ax.set_xticklabels(SIGNAL_NAMES, rotation=45, ha="right", fontsize=7)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1.2); ax.set_ylabel("Hit Rate"); ax.set_ylim(0.3,0.7)
    ax.set_title(f"{p}", fontsize=9)
fig.suptitle("5C — Hit Rate", fontsize=12, y=1.02); fig.tight_layout(); savefig(fig, "5C_hit_rate"); plt.show()

for p in PRODUCTS:
    print(f"\n  {p}:"); res = SIGNAL_RESULTS[p]
    print(f"    {'Signal':15s} {'h1':>7s} {'h5':>7s} {'h20':>7s} {'Hit':>6s}  Verdict")
    for s in sorted(SIGNAL_NAMES, key=lambda s: abs(res[s]["corr"]), reverse=True):
        r=res[s]; d="mom" if r["corr"]>0 else "rev"
        print(f"    {s:15s} {r['corr']:>7.4f} {r['corrs'].get(5,0):>7.4f} {r['corrs'].get(20,0):>7.4f} {r['hit']:>5.1%}  {'USE' if abs(r['corr'])>0.02 else 'skip'} ({d})")

---
# Section 6 — Trade Pattern & Bot Analysis
**Round 1 note:** buyer/seller IDs are empty in sample data. When IDs become available (live trading or later rounds), the counterparty analysis below activates automatically.

Even without IDs, trade sizes, timing, and direction patterns reveal bot behavior.

In [ ]:
# 6: Trade Pattern Analysis
for p in PRODUCTS:
    df = FEAT[p]
    if p not in trade_dfs or len(trade_dfs[p]) == 0:
        print(f"  {p}: No trade data"); continue
    tdf = trade_dfs[p].copy()
    wall_s = df.set_index("timestamp")["wall_mid"]; mid_s = df.set_index("timestamp")["mid_price"]
    def lu(s, t):
        i = s.index.searchsorted(t, side="right")-1
        return s.iloc[i] if i>=0 else np.nan
    tdf["wall_mid"] = tdf["timestamp"].map(lambda t: lu(wall_s, t))
    tdf["mid_at_trade"] = tdf["timestamp"].map(lambda t: lu(mid_s, t))

    # Check for trader IDs
    has_ids = False
    if "buyer" in tdf.columns:
        ids = set(tdf["buyer"].dropna().unique()) | set(tdf["seller"].dropna().unique())
        ids.discard("")
        has_ids = len(ids) > 0
    if has_ids:
        print(f"  {p}: Trader IDs found: {sorted(ids)}")
        # Full counterparty analysis would go here
    else:
        print(f"  {p}: No trader IDs (anonymised) — using trade pattern analysis only")

    # 6C: Trade prices vs FV
    fig, ax = plt.subplots(figsize=(14, 6))
    sm = df["spread"].mean()
    clrs = np.where(tdf["price"]>tdf["wall_mid"]+sm, "orange", np.where(tdf["price"]<tdf["wall_mid"]-sm, "red", "green"))
    ax.scatter(tdf["timestamp"], tdf["price"], c=clrs, s=15, alpha=0.6, edgecolors="none")
    ax.plot(df["timestamp"], df["wall_mid"], color="black", linewidth=0.8, alpha=0.6)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor='green',label='Near FV'), Patch(facecolor='orange',label='Overbid'),
              Patch(facecolor='red',label='Undersell'), plt.Line2D([0],[0],color='black',label='Wall Mid')], fontsize=7)
    ax.set_title(f"{p} — Trades vs FV", fontsize=9); ax.set_xlabel("Timestamp"); ax.set_ylabel("Price")
    fig.tight_layout(); savefig(fig, f"6C_trade_vs_fv_{p}"); plt.show()

    # 6D: Trade sizes
    fig, ax = plt.subplots(figsize=(10, 5)); qtys = tdf["quantity"].dropna()
    ax.hist(qtys, bins=max(10, int(qtys.max()-qtys.min()+1)), color="#5599dd", edgecolor="white", alpha=0.8)
    for sz, cnt in qtys.value_counts().head(5).items():
        ax.annotate(f"qty={sz} (n={cnt})", xy=(sz,cnt), fontsize=7, ha="center", va="bottom",
                    bbox=dict(boxstyle="round,pad=0.2", fc="lightyellow", ec="orange"))
    ax.set_title(f"{p} — Trade sizes (recurring = bot)", fontsize=9); ax.set_xlabel("Qty"); ax.set_ylabel("Freq")
    fig.tight_layout(); savefig(fig, f"6D_trade_sizes_{p}"); plt.show()

    # 6E: Buy/sell pressure
    fig, ax = plt.subplots(figsize=(12, 5))
    tv = tdf.dropna(subset=["mid_at_trade"]).copy()
    tv["buyer_init"] = tv["price"] >= tv["mid_at_trade"]
    bins = np.arange(0, tdf["timestamp"].max()+5000, 5000); bc = (bins[:-1]+bins[1:])/2
    bi = np.clip(np.digitize(tv["timestamp"], bins)-1, 0, len(bins)-2); tv["bin"]=bi
    bcg = tv.groupby("bin")["buyer_init"].sum(); scg = tv.groupby("bin")["buyer_init"].apply(lambda x:(~x).sum())
    bcc = [bcg.get(b,0) for b in range(len(bins)-1)]; scc = [scg.get(b,0) for b in range(len(bins)-1)]
    ax.bar(bc, bcc, width=4500, color="green", alpha=0.7, label="Buy-init")
    ax.bar(bc, scc, width=4500, bottom=bcc, color="red", alpha=0.7, label="Sell-init")
    ax.set_title(f"{p} — Trade direction", fontsize=9); ax.set_xlabel("Timestamp"); ax.set_ylabel("Count"); ax.legend(fontsize=7)
    fig.tight_layout(); savefig(fig, f"6E_trade_direction_{p}"); plt.show()

    # 6F: Trade flow → future price impact
    fig, ax = plt.subplots(figsize=(12, 5))
    tv_valid = tv.copy()
    tv_valid["signed_qty"] = np.where(tv_valid["buyer_init"], tv_valid["quantity"], -tv_valid["quantity"])
    # Rolling net flow in 2000-tick window
    ts_range = df["timestamp"].values[::20]
    net_flow = []
    for t in ts_range:
        window = tv_valid[(tv_valid["timestamp"] >= t-2000) & (tv_valid["timestamp"] <= t)]
        net_flow.append(window["signed_qty"].sum())
    net_flow = np.array(net_flow)
    mid_at_ts = df.drop_duplicates("timestamp").set_index("timestamp")["mid_price"].sort_index().reindex(ts_range, method="ffill")
    fwd_ret = mid_at_ts.diff(5).shift(-5)
    ax2 = ax.twinx()
    ax.plot(ts_range, net_flow, alpha=0.7, linewidth=0.6, color="blue", label="Net flow (2k window)")
    ax2.plot(ts_range, fwd_ret, alpha=0.5, linewidth=0.5, color="red", label="Fwd 5-tick ret")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_xlabel("Timestamp"); ax.set_ylabel("Net Flow (buy−sell)", color="blue")
    ax2.set_ylabel("Forward Return", color="red")
    ax.set_title(f"{p} — Does order flow predict price? (flow leads return = alpha)", fontsize=9)
    ax.legend(loc="upper left", fontsize=7); ax2.legend(loc="upper right", fontsize=7)
    fig.tight_layout(); savefig(fig, f"6F_flow_impact_{p}"); plt.show()

    print(f"  {p:30s}  trades={len(tdf)}  buyer_ratio={tv['buyer_init'].mean():.1%}" if len(tv)>0 else f"  {p}: no valid trades")

---
# Section 7 — Intraday Pattern Detection
**Critical for ASH_COATED_OSMIUM** — the "hidden pattern" hint suggests a daily cycle.
Overlays all days, computes average intraday path, and checks for periodicity via long-lag ACF.

In [ ]:
# 7A: All days overlaid
for p in PRODUCTS:
    df = FEAT[p]; days = sorted(df["day"].unique())
    fig, ax = plt.subplots(figsize=(14, 6))
    for di, day in enumerate(days):
        dd = df[df["day"]==day].copy()
        ax.plot(dd["timestamp"].values, dd["mid_price"].values - dd["mid_price"].values[0],
                color=DAY_COLORS[di%len(DAY_COLORS)], alpha=0.7, linewidth=0.8, label=f"Day {day}")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"{p} — Price change from open: do days share a pattern?", fontsize=9)
    ax.set_xlabel("Timestamp"); ax.set_ylabel("Price − Day Open"); ax.legend(fontsize=7)
    fig.tight_layout(); savefig(fig, f"7A_intraday_overlay_{p}"); plt.show()

In [ ]:
# 7B: Average intraday path
for p in PRODUCTS:
    df = FEAT[p]; days = sorted(df["day"].unique())
    fig, ax = plt.subplots(figsize=(14, 6))
    n_bins = 100; all_paths = []
    for di, day in enumerate(days):
        dd = df[df["day"]==day]; ts = dd["timestamp"].values; pr = dd["mid_price"].values - dd["mid_price"].values[0]
        if len(dd) < 10: continue
        be = np.linspace(ts.min(), ts.max(), n_bins+1); bi = np.clip(np.digitize(ts, be)-1, 0, n_bins-1)
        bm = np.array([pr[bi==b].mean() if (bi==b).any() else np.nan for b in range(n_bins)])
        all_paths.append(bm)
        ax.plot(np.linspace(0, 1, n_bins), bm, alpha=0.3, linewidth=0.5, color="gray")
    if len(all_paths) > 1:
        avg = np.nanmean(all_paths, axis=0); std = np.nanstd(all_paths, axis=0); x = np.linspace(0,1,n_bins)
        ax.plot(x, avg, color="black", linewidth=2.5, label="Average")
        ax.fill_between(x, avg-std, avg+std, alpha=0.2, color="gray", label="±1σ")
    ax.axhline(0, color="red", linewidth=0.5, linestyle="--")
    ax.set_title(f"{p} — Average intraday path (0=open, 1=close)", fontsize=9)
    ax.set_xlabel("Fraction of day"); ax.set_ylabel("Price Δ"); ax.legend(fontsize=7)
    fig.tight_layout(); savefig(fig, f"7B_avg_intraday_path_{p}"); plt.show()

In [ ]:
# 7C: Long-lag ACF (periodicity)
for p in PRODUCTS:
    df = FEAT[p]; rets = df["ret"].dropna()
    fig, ax = plt.subplots(figsize=(12, 5))
    ll = list(range(1, min(201, len(rets)//5)))
    acfs = [rets.autocorr(lag=l) for l in ll]
    ax.plot(ll, acfs, alpha=0.7, linewidth=0.8, color="#1f77b4"); ax.axhline(0, color="black", linewidth=0.5)
    sig = 2/np.sqrt(len(rets))
    ax.axhline(sig, color="red", linestyle=":", linewidth=0.8, label=f"Sig ±{sig:.4f}")
    ax.axhline(-sig, color="red", linestyle=":", linewidth=0.8)
    acfs_a = np.array(acfs)
    peaks = [(ll[i], acfs_a[i]) for i in range(1, len(acfs_a)-1)
             if abs(acfs_a[i])>sig and abs(acfs_a[i])>abs(acfs_a[i-1]) and abs(acfs_a[i])>abs(acfs_a[i+1])]
    for lag, val in peaks[:5]:
        ax.annotate(f"lag={lag}", xy=(lag,val), fontsize=7, ha="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="lightyellow", ec="orange"))
    ax.set_title(f"{p} — Long-lag ACF: peaks = periodicity", fontsize=9)
    ax.set_xlabel("Lag"); ax.set_ylabel("ACF"); ax.legend(fontsize=7)
    fig.tight_layout(); savefig(fig, f"7C_periodicity_{p}"); plt.show()
    if peaks: print(f"  {p}: periodicities at {[(l,f'{v:.4f}') for l,v in peaks[:5]]}")
    else: print(f"  {p}: no significant periodicity")

---
# Section 8 — Spike Detection & Event Study
Detects spikes (>3x rolling vol) and tests if they revert — basis for spike-fade trading.

In [ ]:
SPIKE_RESULTS = {}
for p in PRODUCTS:
    df = FEAT[p]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax1.plot(df["timestamp"], df["mid_price"], alpha=0.7, linewidth=0.5, color="#1f77b4")
    up = df[df["spike"]&(df["ret"]>0)]; dn = df[df["spike"]&(df["ret"]<0)]
    ax1.scatter(up["timestamp"], up["mid_price"], marker="v", color="red", s=30, alpha=0.7, label="Up spike")
    ax1.scatter(dn["timestamp"], dn["mid_price"], marker="^", color="green", s=30, alpha=0.7, label="Down spike")
    ax1.set_ylabel("Mid"); ax1.legend(fontsize=7)
    ax2.plot(df["timestamp"], df["rolling_vol"], alpha=0.7, linewidth=0.5, color="#ff7f0e")
    mv = df["rolling_vol"].mean()
    ax2.axhline(3*mv, color="red", linestyle="--", linewidth=1, label=f"3x={3*mv:.3f}")
    ax2.set_ylabel("Vol"); ax2.set_xlabel("Timestamp"); ax2.legend(fontsize=7)
    ns = df["spike"].sum()
    fig.suptitle(f"8A — {p}: {ns} spikes in {len(df)} ticks", fontsize=10, y=1.01)
    fig.tight_layout(); savefig(fig, f"8A_spikes_{p}"); plt.show()

In [ ]:
# 8B: Post-spike reversion
for p in PRODUCTS:
    df = FEAT[p]; up = df[df["spike"]&(df["ret"]>0)]; dn = df[df["spike"]&(df["ret"]<0)]
    fig, ax = plt.subplots(figsize=(12, 6)); h = 20
    for label, idx, color in [("After UP", up.index.tolist(), "red"), ("After DOWN", dn.index.tolist(), "green")]:
        paths = [df["mid_price"].iloc[i:i+h+1].values - df["mid_price"].iloc[i] for i in idx if i+h<len(df)]
        if len(paths) > 2:
            paths = np.array(paths); m = paths.mean(axis=0); s = paths.std(axis=0)
            ax.plot(range(h+1), m, color=color, linewidth=2, label=label)
            ax.fill_between(range(h+1), m-s, m+s, alpha=0.15, color=color)
    ax.axhline(0, color="black", linewidth=0.5); ax.set_xlabel("Ticks after spike"); ax.set_ylabel("Cum return")
    ax.set_title(f"{p} — Post-spike reversion", fontsize=9); ax.legend(fontsize=8)
    fig.tight_layout(); savefig(fig, f"8B_spike_reversion_{p}"); plt.show()

    ur = np.mean([df["mid_price"].iloc[i+5]-df["mid_price"].iloc[i] for i in up.index if i+5<len(df)]) if len(up)>0 else 0
    dr = np.mean([df["mid_price"].iloc[i+5]-df["mid_price"].iloc[i] for i in dn.index if i+5<len(df)]) if len(dn)>0 else 0
    rev = (ur<0 and len(up)>5) or (dr>0 and len(dn)>5)
    SPIKE_RESULTS[p] = {"n_spikes":df["spike"].sum(), "freq_per_10k":df["spike"].sum()/len(df)*10000,
                        "avg_mag":df.loc[df["spike"],"ret"].abs().mean() if df["spike"].sum()>0 else 0, "reversion":rev}
    print(f"  {p:30s}  spikes/10k={SPIKE_RESULTS[p]['freq_per_10k']:.1f}  up5={ur:.3f}  dn5={dr:.3f}  → {'REVERSION' if rev else 'NO'}")

---
# Section 9 — Cross-Product Analysis
Correlation and lead-lag between ASH_COATED_OSMIUM and INTARIAN_PEPPER_ROOT.

In [ ]:
if len(PRODUCTS) >= 2:
    ret_dict = {p: FEAT[p].set_index(["day","timestamp"])["mid_price"] for p in PRODUCTS}
    aligned = pd.DataFrame(ret_dict).dropna(); rets = aligned.diff().dropna()

    fig, ax = plt.subplots(figsize=(14, 6))
    for idx, p in enumerate(PRODUCTS):
        normed = (aligned[p]-aligned[p].mean())/aligned[p].std()
        ax.plot(normed.values, alpha=0.7, linewidth=0.6, color=DAY_COLORS[idx], label=p)
    ax.set_title("9A — Normalised prices"); ax.legend(); fig.tight_layout()
    savefig(fig, "9A_normalised_prices"); plt.show()

    corr_mat = rets.corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr_mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(PRODUCTS))); ax.set_xticklabels(PRODUCTS, rotation=45, ha="right")
    ax.set_yticks(range(len(PRODUCTS))); ax.set_yticklabels(PRODUCTS)
    for i in range(len(PRODUCTS)):
        for j in range(len(PRODUCTS)):
            ax.text(j, i, f"{corr_mat.iloc[i,j]:.3f}", ha="center", va="center", fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8); ax.set_title("9B — Correlation"); fig.tight_layout()
    savefig(fig, "9B_correlation_matrix"); plt.show()

    pairs = list(itertools.combinations(PRODUCTS, 2))
    if pairs:
        fig, axes = plt.subplots(1, len(pairs), figsize=(10*len(pairs), 5), squeeze=False)
        for idx, (p1,p2) in enumerate(pairs):
            ax = axes[0,idx]; r1=rets[p1].values; r2=rets[p2].values; lr=range(-10,11)
            xc = [np.corrcoef(r1[:len(r1)-abs(l)] if l>=0 else r1[abs(l):], r2[abs(l):] if l>=0 else r2[:len(r2)-abs(l)])[0,1] for l in lr]
            ax.bar(list(lr), xc, color=["#1f77b4" if abs(x)>0.05 else "#ccc" for x in xc], edgecolor="white")
            ax.axhline(0, color="black", linewidth=0.5); ax.set_xlabel(f"Lag (+={p1} leads)"); ax.set_ylabel("XCorr")
            ax.set_title(f"Lead-lag", fontsize=9)
        fig.tight_layout(); savefig(fig, "9C_lead_lag"); plt.show()
    print("Correlation:"); print(corr_mat.to_string(float_format=lambda x: f"{x:.4f}"))
else:
    print("Only 1 product — skipping")

---
# Section 10 — Position & Inventory Risk
Simulates take/clear strategy and tracks position + mark-to-market PnL.

In [ ]:
POS_RESULTS = {}
for p in PRODUCTS:
    df = FEAT[p]; te = SPREAD_RESULTS.get(p,{}).get("take_edge",1.0)
    limit = POSITION_LIMITS.get(p, DEFAULT_LIMIT)
    fv = df["wall_mid"].values; a1 = df["ask_price_1"].values; b1 = df["bid_price_1"].values
    av1 = df["ask_volume_1"].fillna(0).values; bv1 = df["bid_volume_1"].fillna(0).values
    pos = 0; positions = []; pnl = 0; pnls = []
    for i in range(len(df)):
        if a1[i] < fv[i]-te and pos < limit:
            q = min(int(av1[i]), limit-pos, 10); pos += q; pnl -= a1[i]*q
        if b1[i] > fv[i]+te and pos > -limit:
            q = min(int(bv1[i]), limit+pos, 10); pos -= q; pnl += b1[i]*q
        if abs(pos) > limit*0.5:
            cq = min(abs(pos)-int(limit*0.3), 3)
            if pos>0 and b1[i]>=fv[i]-1: sq=min(cq,pos); pos-=sq; pnl+=b1[i]*sq
            elif pos<0 and a1[i]<=fv[i]+1: sq=min(cq,-pos); pos+=sq; pnl-=a1[i]*sq
        positions.append(pos); pnls.append(pnl + pos*fv[i])
    positions = np.array(positions); pnls = np.array(pnls)

    fig, (ax1,ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax1.plot(df["timestamp"], positions, alpha=0.7, linewidth=0.5, color="#1f77b4")
    ax1.axhline(limit, color="red", linestyle="--"); ax1.axhline(-limit, color="red", linestyle="--")
    ax1.axhline(limit*0.75, color="orange", linestyle=":"); ax1.axhline(-limit*0.75, color="orange", linestyle=":")
    ax1.set_title(f"{p} — Position (TE={te:.2f})", fontsize=9); ax1.set_ylabel("Position")
    ax2.plot(df["timestamp"], pnls, alpha=0.7, linewidth=0.8, color="#2ca02c")
    ax2.set_title("MtM PnL", fontsize=9); ax2.set_xlabel("Timestamp"); ax2.set_ylabel("PnL")
    fig.tight_layout(); savefig(fig, f"10A_position_pnl_{p}"); plt.show()

    POS_RESULTS[p] = {"pct_at_limit":(np.abs(positions)>=limit*0.9).mean()*100, "final_pnl":pnls[-1], "max_pos":int(np.max(np.abs(positions)))}
    print(f"  {p:30s}  %lim={POS_RESULTS[p]['pct_at_limit']:.1f}%  max={POS_RESULTS[p]['max_pos']}  pnl={POS_RESULTS[p]['final_pnl']:.0f}")

---
# Section 11 — Parameter Sensitivity
Grid over take_edge × clear_threshold. Top teams emphasised stability over peak.

In [ ]:
for p in PRODUCTS:
    df = FEAT[p]; limit = POSITION_LIMITS.get(p, DEFAULT_LIMIT)
    fv = df["wall_mid"].values; a1 = df["ask_price_1"].values; b1 = df["bid_price_1"].values
    av1 = df["ask_volume_1"].fillna(0).values; bv1 = df["bid_volume_1"].fillna(0).values
    te_r = np.arange(0.5, 6.1, 0.5); ct_r = [0.3, 0.5, 0.7]
    grid = np.zeros((len(te_r), len(ct_r)))
    for ti, te in enumerate(te_r):
        for ci, ct in enumerate(ct_r):
            pos = 0; pnl = 0
            for i in range(len(df)):
                if a1[i]<fv[i]-te and pos<limit: q=min(int(av1[i]),limit-pos,10); pos+=q; pnl-=a1[i]*q
                if b1[i]>fv[i]+te and pos>-limit: q=min(int(bv1[i]),limit+pos,10); pos-=q; pnl+=b1[i]*q
                if abs(pos)>limit*ct:
                    cq=min(abs(pos)-int(limit*0.3),3)
                    if pos>0 and b1[i]>=fv[i]-1: sq=min(cq,pos); pos-=sq; pnl+=b1[i]*sq
                    elif pos<0 and a1[i]<=fv[i]+1: sq=min(cq,-pos); pos+=sq; pnl-=a1[i]*sq
            grid[ti,ci] = pnl + pos*fv[-1]
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(grid, cmap="RdYlGn", aspect="auto", origin="lower")
    ax.set_xticks(range(len(ct_r))); ax.set_xticklabels([f"{c:.0%}" for c in ct_r])
    ax.set_yticks(range(len(te_r))); ax.set_yticklabels([f"{t:.1f}" for t in te_r])
    ax.set_xlabel("Clear Threshold"); ax.set_ylabel("Take Edge")
    for i in range(len(te_r)):
        for j in range(len(ct_r)):
            ax.text(j, i, f"{grid[i,j]:.0f}", ha="center", va="center", fontsize=7)
    plt.colorbar(im, ax=ax, shrink=0.8, label="MtM PnL")
    bi = np.unravel_index(grid.argmax(), grid.shape)
    ax.set_title(f"{p} — Best: TE={te_r[bi[0]]:.1f} CT={ct_r[bi[1]]:.0%} PnL={grid[bi]:.0f}", fontsize=9)
    fig.tight_layout(); savefig(fig, f"11A_param_grid_{p}"); plt.show()
    print(f"  {p:30s}  best_TE={te_r[bi[0]]:.1f}  best_CT={ct_r[bi[1]]:.0%}  pnl={grid[bi]:.0f}")

---
# Section 12 — Strategy Decision Dashboard
Consolidated per-product dashboard with final recommendation.

In [ ]:
strat_map = {"STABLE":"Fixed FV MM (~10000)","DRIFTING":"Dynamic FV MM","VOLATILE":"Spike Rev + Wide MM"}
for p in PRODUCTS:
    df = FEAT[p]; v = VERDICTS.get(p,"STABLE")
    sr = SPREAD_RESULTS.get(p,{}); sig = SIGNAL_RESULTS.get(p,{})
    spk = SPIKE_RESULTS.get(p,{}); fv = FV_RESULTS.get(p,{})
    te = sr.get("take_edge",1); pe = sr.get("passive_edge",1)
    limit = POSITION_LIMITS.get(p, DEFAULT_LIMIT)

    fig, axes = plt.subplots(3, 3, figsize=(22, 18))
    ax=axes[0,0]
    for di, day in enumerate(sorted(df["day"].unique())):
        dd = df[df["day"]==day]
        ax.plot(dd["timestamp"], dd["mid_price"], color=DAY_COLORS[di%len(DAY_COLORS)], alpha=0.7, linewidth=0.5)
    ax.set_title(f"Price → {v}", fontsize=9)

    ax=axes[0,1]; rets=df["ret"].dropna()
    av=[rets.autocorr(lag=l) for l in [1,2,3,5,10]]
    ax.bar(range(5), av, color=["#d62728" if a<0 else "#1f77b4" for a in av], edgecolor="white")
    ax.set_xticks(range(5)); ax.set_xticklabels(["1","2","3","5","10"]); ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"ACF → {'MR' if av[0]<-0.05 else ('TR' if av[0]>0.05 else 'N')}", fontsize=9)

    ax=axes[0,2]; ax.plot(df["timestamp"], df["z20"], alpha=0.7, linewidth=0.5, color="#ff7f0e")
    for lv in [1,-1,2,-2]: ax.axhline(lv, color="gray", linestyle="--" if abs(lv)==1 else ":", linewidth=0.7)
    ax.axhline(0, color="black", linewidth=0.5); ax.set_title("Z-Score", fontsize=9)

    ax=axes[1,0]; sp=df["spread"].dropna()
    ax.hist(sp, bins=50, color="#5599dd", edgecolor="white", alpha=0.8)
    ax.axvline(sp.median(), color="orange", linestyle="--"); ax.set_title(f"Spread (TE={te:.2f} PE={pe:.2f})", fontsize=9)

    ax=axes[1,1]
    sn=["ret1","z20","micro_delta","wall_delta","mm_delta","imbalance","spread_chg"]
    if sig:
        ss2=sorted(sn, key=lambda s: abs(sig.get(s,{}).get("corr",0)))
        cs2=[sig.get(s,{}).get("corr",0) for s in ss2]
        ax.barh(range(len(ss2)), cs2, color=["#d62728" if c<0 else "#1f77b4" for c in cs2], edgecolor="white")
        ax.set_yticks(range(len(ss2))); ax.set_yticklabels(ss2, fontsize=7)
        ax.set_title(f"Signals → {ss2[-1]}", fontsize=9)
    ax.axvline(0, color="black", linewidth=0.5)

    ax=axes[1,2]; ax.plot(df["timestamp"], df["mid_price"], alpha=0.6, linewidth=0.5, color="#1f77b4")
    us=df[df["spike"]&(df["ret"]>0)]; ds2=df[df["spike"]&(df["ret"]<0)]
    ax.scatter(us["timestamp"], us["mid_price"], marker="v", color="red", s=20, alpha=0.7)
    ax.scatter(ds2["timestamp"], ds2["mid_price"], marker="^", color="green", s=20, alpha=0.7)
    ax.set_title(f"Spikes → Rev: {'YES' if spk.get('reversion') else 'NO'}", fontsize=9)

    ax=axes[2,0]; step=max(1,len(df)//2000); d2=df.iloc[::step]
    ax.plot(d2["timestamp"], d2["wall_mid"], color="green", linewidth=0.8, alpha=0.7, label="Wall")
    ax.plot(d2["timestamp"], d2["mm_mid"], color="red", linewidth=0.8, alpha=0.7, label="MM")
    best_fv = min(fv, key=fv.get).replace("_std","") if fv else "wall_mid"
    ax.set_title(f"FV → {best_fv}", fontsize=9); ax.legend(fontsize=6)

    ax=axes[2,1]; pos2=0; pa2=[]
    fva=df["wall_mid"].values; aa=df["ask_price_1"].values; bb=df["bid_price_1"].values
    aav=df["ask_volume_1"].fillna(0).values; bbv=df["bid_volume_1"].fillna(0).values
    for i in range(len(df)):
        if aa[i]<fva[i]-te and pos2<limit: pos2+=min(int(aav[i]),limit-pos2,5)
        if bb[i]>fva[i]+te and pos2>-limit: pos2-=min(int(bbv[i]),limit+pos2,5)
        pa2.append(pos2)
    pa2=np.array(pa2)
    ax.plot(df["timestamp"], pa2, alpha=0.7, linewidth=0.5, color="#1f77b4")
    ax.axhline(limit, color="red", linestyle="--"); ax.axhline(-limit, color="red", linestyle="--")
    ax.set_title(f"Position (lim%={(np.abs(pa2)>=limit*0.9).mean()*100:.1f}%)", fontsize=9)

    ax=axes[2,2]; ax.axis("off")
    bs = max(sn, key=lambda s: abs(sig.get(s,{}).get("corr",0))) if sig else "N/A"
    bc = sig.get(bs,{}).get("corr",0) if sig else 0
    txt = f"{'━'*40}\n  FINAL RECOMMENDATION\n{'━'*40}\n\n  Type: {v}\n  Strategy: {strat_map.get(v,'MM')}\n  Fair Value: {best_fv}\n  Take Edge: {te:.2f}\n  Passive Edge: {pe:.2f}\n  Signal: {bs} ({bc:.3f})\n  Spike Rev: {'YES' if spk.get('reversion') else 'NO'}\n  Pos Warn: {limit*0.75:.0f}"
    ax.text(0.05, 0.95, txt, transform=ax.transAxes, fontsize=11, verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.5", fc="#ffffdd", ec="#cc8800", linewidth=2))
    fig.suptitle(f"{p} — Dashboard", fontsize=14, y=1.01); fig.tight_layout()
    savefig(fig, f"12_dashboard_{p}"); plt.show()
    print(f"\n  {p}: {v} → {strat_map.get(v,'MM')}, FV={best_fv}, TE={te:.2f}, PE={pe:.2f}, Sig={bs}({bc:.3f})")

---
## Export — Download All Plots & Analysis


In [ ]:
import shutil, datetime, io, contextlib

# Create timestamped export folder
export_name = f"analysis_export_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
export_dir = os.path.join(os.path.dirname(os.path.abspath(PLOT_DIR)), export_name)
os.makedirs(export_dir, exist_ok=True)

# 1) Copy all plots
plots_export = os.path.join(export_dir, "plots")
if os.path.exists(PLOT_DIR):
    shutil.copytree(PLOT_DIR, plots_export)
    plot_count = len([f for f in os.listdir(plots_export) if f.endswith(".png")])
else:
    plot_count = 0

# 2) Generate text summary by capturing print output from key sections
summary_lines = []
summary_lines.append("IMC Prosperity 4 — Round 1 Analysis Summary")
summary_lines.append(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("=" * 60)

summary_lines.append(f"\nProducts: {PRODUCTS}")
summary_lines.append(f"Days: {sorted(prices['day'].unique())}")
summary_lines.append(f"Total price rows: {len(prices):,}")
summary_lines.append(f"Total trade rows: {len(trades):,}")

for p in PRODUCTS:
    df = FEAT[p]
    summary_lines.append(f"\n{'='*60}")
    summary_lines.append(f"  {p}")
    summary_lines.append(f"{'='*60}")

    # Basic stats
    summary_lines.append(f"  Ticks: {len(df):,}")
    summary_lines.append(f"  Price range: [{df['mid_price'].min():.0f}, {df['mid_price'].max():.0f}]")
    summary_lines.append(f"  Mean: {df['mid_price'].mean():.1f}, Std: {df['mid_price'].std():.1f}")
    summary_lines.append(f"  Spread: mean={df['spread'].mean():.1f}, median={df['spread'].median():.1f}")

    # Classification
    v = VERDICTS.get(p, "?")
    summary_lines.append(f"  Classification: {v}")

    # Spread / edges
    sr = SPREAD_RESULTS.get(p, {})
    summary_lines.append(f"  Take edge: {sr.get('take_edge', '?')}")
    summary_lines.append(f"  Passive edge: {sr.get('passive_edge', '?')}")

    # ACF
    acf = ACF_RESULTS.get(p, {})
    lag1 = acf.get('lag1', 0)
    mr = "MEAN REVERTING" if lag1 < -0.05 else ("TRENDING" if lag1 > 0.05 else "NEUTRAL")
    summary_lines.append(f"  Lag-1 ACF: {lag1:.4f} -> {mr}")

    # Signals
    sig = SIGNAL_RESULTS.get(p, {})
    if sig:
        best_sig = max(SIGNAL_NAMES, key=lambda s: abs(sig.get(s, {}).get("corr", 0)))
        best_corr = sig[best_sig]["corr"]
        summary_lines.append(f"  Best signal: {best_sig} (corr={best_corr:.4f})")

    # Spikes
    spk = SPIKE_RESULTS.get(p, {})
    summary_lines.append(f"  Spike reversion: {'YES' if spk.get('reversion') else 'NO'}")

    # Position
    pos = POS_RESULTS.get(p, {})
    summary_lines.append(f"  Sim PnL: {pos.get('final_pnl', 0):.0f}, %at limit: {pos.get('pct_at_limit', 0):.1f}%")

summary_text = "\n".join(summary_lines)

with open(os.path.join(export_dir, "analysis_summary.txt"), "w") as f:
    f.write(summary_text)

# 3) Copy the raw data files for reproducibility
data_export = os.path.join(export_dir, "data")
os.makedirs(data_export, exist_ok=True)
import glob as glob_mod
for pattern in ["prices_round_*_day_*.csv", "trades_round_*_day_*.csv"]:
    for fp in glob_mod.glob(pattern):
        shutil.copy2(fp, data_export)
data_count = len(os.listdir(data_export))

# 4) Zip the entire export folder
zip_path = shutil.make_archive(export_dir, 'zip', os.path.dirname(export_dir), export_name)

print(f"Export complete!")
print(f"  Folder : {export_dir}/")
print(f"  Zip    : {zip_path}")
print(f"  Plots  : {plot_count} PNGs")
print(f"  Data   : {data_count} CSVs")
print(f"  Summary: analysis_summary.txt")


---
## AI Strategy Export — Context-Optimized for Codex / Claude Code
Curated subset of plots + structured data brief. Optimized for token efficiency:
images are expensive (~1500 tokens each), so only include plots where visual pattern
recognition adds value beyond what numbers capture.


In [ ]:
import shutil, datetime, json as json_mod

ai_export_dir = os.path.join(os.path.dirname(os.path.abspath(PLOT_DIR)), "ai_strategy_context")
if os.path.exists(ai_export_dir):
    shutil.rmtree(ai_export_dir)
os.makedirs(ai_export_dir, exist_ok=True)

# ─────────────────────────────────────────────────────────
# 1) STRATEGY BRIEF (markdown) — primary artifact for AI
#    This replaces ~40 plots worth of information as text
# ─────────────────────────────────────────────────────────
brief_lines = []
brief_lines.append("# Round 1 Strategy Brief")
brief_lines.append(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
brief_lines.append("## Data Overview")
brief_lines.append(f"- Days available: {sorted(prices['day'].unique())}")
brief_lines.append(f"- Price ticks: {len(prices):,} total (after filtering empty books)")
brief_lines.append(f"- Trades: {len(trades):,} total")
brief_lines.append(f"- Buyer/seller IDs: **empty** (anonymised in Round 1 sample data)\n")

strat_map = {"STABLE": "Fixed Fair Value Market Making",
             "DRIFTING": "Dynamic / Rolling Fair Value Market Making",
             "VOLATILE": "Spike Reversion + Wide-edge Market Making"}

for p in PRODUCTS:
    df = FEAT[p]
    v = VERDICTS.get(p, "?")
    sr = SPREAD_RESULTS.get(p, {})
    acf = ACF_RESULTS.get(p, {})
    sig = SIGNAL_RESULTS.get(p, {})
    spk = SPIKE_RESULTS.get(p, {})
    pos = POS_RESULTS.get(p, {})
    limit = POSITION_LIMITS.get(p, 80)

    # Classification details
    _, avg_cv, drift_val, has_drift = classify_product_within_day(df)

    # Best FV
    fv_stds = {}
    for method, col in [("simple_mid", "simple_mid"), ("microprice", "microprice"),
                         ("wall_mid", "wall_mid"), ("mm_mid", "mm_mid")]:
        fv_stds[method] = df[col].std()
    best_fv = min(fv_stds, key=fv_stds.get)

    # Best signal
    signal_names_local = ["ret1", "z20", "micro_delta", "wall_delta", "mm_delta", "imbalance", "spread_chg"]
    if sig:
        best_sig = max(signal_names_local, key=lambda s: abs(sig.get(s, {}).get("corr", 0)))
        best_corr = sig[best_sig]["corr"]
        best_hit = sig[best_sig]["hit"]
    else:
        best_sig, best_corr, best_hit = "N/A", 0, 0.5

    # ACF
    lag1 = acf.get("lag1", 0)
    mr_label = "MEAN REVERTING" if lag1 < -0.05 else ("TRENDING" if lag1 > 0.05 else "NEUTRAL")

    # Daily stats
    days = sorted(df["day"].unique())
    day_means = [df[df["day"] == d]["mid_price"].mean() for d in days]

    brief_lines.append(f"\n---\n## {p}")
    brief_lines.append(f"**Position limit:** {limit}")
    brief_lines.append(f"**Classification:** {v} (within-day CV={avg_cv:.6f})")
    if has_drift:
        brief_lines.append(f"**Drift:** {drift_val:.0f}/day ({'upward' if day_means[-1] > day_means[0] else 'downward'})")
    brief_lines.append(f"**Recommended strategy:** {strat_map.get(v, 'MM')}\n")

    brief_lines.append("### Price Behavior")
    for di, day in enumerate(days):
        dd = df[df["day"] == day]["mid_price"]
        brief_lines.append(f"- Day {day}: open={dd.iloc[0]:.0f}, close={dd.iloc[-1]:.0f}, "
                          f"low={dd.min():.0f}, high={dd.max():.0f}, mean={dd.mean():.0f}, std={dd.std():.1f}")
    brief_lines.append(f"- Overall mean: {df['mid_price'].mean():.1f}, overall std: {df['mid_price'].std():.1f}")
    brief_lines.append(f"- Return autocorrelation: lag-1={lag1:.4f} → **{mr_label}**")
    if v == "STABLE":
        brief_lines.append(f"- Price anchored at ~{round(df['mid_price'].mean()/100)*100}. Deviations revert.")
    elif v == "DRIFTING":
        brief_lines.append(f"- Fair value shifts ~{drift_val:.0f} per day. Must track with rolling window.")

    brief_lines.append(f"\n### Spread & Edge Parameters")
    brief_lines.append(f"- Spread: mean={sr.get('mean', 0):.1f}, median (p50)={sr.get('p50', 0):.1f}, "
                      f"p25={sr.get('p25', 0):.1f}, p75={sr.get('p75', 0):.1f}")
    brief_lines.append(f"- **take_edge = {sr.get('take_edge', 0):.2f}** (= p25/4, sweep when ask < FV - take_edge)")
    brief_lines.append(f"- **passive_edge = {sr.get('passive_edge', 0):.2f}** (= p50/3, quote at FV ± passive_edge)")

    brief_lines.append(f"\n### Fair Value Estimator")
    brief_lines.append(f"- **Best: {best_fv}** (lowest std = most stable)")
    for method in ["simple_mid", "microprice", "wall_mid", "mm_mid"]:
        marker = " ← USE" if method == best_fv else ""
        brief_lines.append(f"  - {method}: std={fv_stds[method]:.4f}{marker}")
    if best_fv == "mm_mid":
        brief_lines.append(f"- MM-filtered mid uses orders >= {MM_SIZE_THRESHOLD} lots (filters noise from small orders)")
    elif best_fv == "wall_mid":
        brief_lines.append(f"- Wall mid = midpoint of largest-volume bid and ask levels")

    brief_lines.append(f"\n### Signal Predictiveness (correlation with next-tick return)")
    brief_lines.append(f"| Signal | Corr(h=1) | Corr(h=5) | Corr(h=20) | Hit Rate | Direction |")
    brief_lines.append(f"|--------|-----------|-----------|------------|----------|-----------|")
    if sig:
        for s in sorted(signal_names_local, key=lambda s: abs(sig.get(s, {}).get("corr", 0)), reverse=True):
            r = sig[s]
            c1 = r["corr"]; c5 = r["corrs"].get(5, 0); c20 = r["corrs"].get(20, 0)
            direction = "momentum" if c1 > 0 else "contrarian"
            use = "**USE**" if abs(c1) > 0.02 else "skip"
            brief_lines.append(f"| {s} | {c1:.4f} | {c5:.4f} | {c20:.4f} | {r['hit']:.1%} | {direction} {use} |")
    brief_lines.append(f"\n**Best signal: {best_sig}** (corr={best_corr:.4f}, hit={best_hit:.1%})")

    brief_lines.append(f"\n### Spike Behavior")
    brief_lines.append(f"- Spikes detected: {spk.get('n_spikes', 0)} ({spk.get('freq_per_10k', 0):.1f} per 10k ticks)")
    brief_lines.append(f"- Average spike magnitude: {spk.get('avg_mag', 0):.2f}")
    brief_lines.append(f"- **Reversion tradeable: {'YES' if spk.get('reversion') else 'NO'}**")

    brief_lines.append(f"\n### Position & Risk (simulated take-only strategy)")
    brief_lines.append(f"- Max position reached: {pos.get('max_pos', 0)}")
    brief_lines.append(f"- Time at limit (≥90%): {pos.get('pct_at_limit', 0):.1f}%")
    brief_lines.append(f"- Simulated MtM PnL: {pos.get('final_pnl', 0):.0f}")

    # Periodicity (from section 7)
    rets = df["ret"].dropna()
    long_lags = list(range(1, min(201, len(rets) // 5)))
    acfs_long = [rets.autocorr(lag=l) for l in long_lags]
    sig_thresh = 2 / np.sqrt(len(rets))
    acfs_arr = np.array(acfs_long)
    peaks = []
    for i in range(1, len(acfs_arr) - 1):
        if (abs(acfs_arr[i]) > sig_thresh and
            abs(acfs_arr[i]) > abs(acfs_arr[i-1]) and
            abs(acfs_arr[i]) > abs(acfs_arr[i+1])):
            peaks.append((long_lags[i], acfs_arr[i]))
    if peaks:
        brief_lines.append(f"\n### Periodicities (significant ACF peaks)")
        for lag, val in peaks[:8]:
            brief_lines.append(f"- Lag {lag}: ACF={val:.4f}")
        brief_lines.append(f"- These could indicate bot trading cycles. Consider using as timing signals.")

# Cross-product
if len(PRODUCTS) >= 2:
    brief_lines.append(f"\n---\n## Cross-Product Relationships")
    ret_dict = {}
    for p in PRODUCTS:
        s = FEAT[p].set_index(["day", "timestamp"])["mid_price"]
        ret_dict[p] = s
    aligned = pd.DataFrame(ret_dict).dropna()
    rets_cross = aligned.diff().dropna()
    corr_mat = rets_cross.corr()
    for p1 in PRODUCTS:
        for p2 in PRODUCTS:
            if p1 < p2:
                c = corr_mat.loc[p1, p2]
                brief_lines.append(f"- {p1} ↔ {p2}: correlation = {c:.4f} "
                                  f"({'correlated' if abs(c) > 0.3 else 'uncorrelated — trade independently'})")

# Implementation notes
brief_lines.append(f"\n---\n## Implementation Notes")
brief_lines.append(f"- Execution model: Trader class, `run()` called each tick with TradingState")
brief_lines.append(f"- Stateless (AWS Lambda): persist state via `traderData` (str, 50k char limit)")
brief_lines.append(f"- Order types: limit orders only, via `{{product: [Order(symbol, price, quantity)]}}` dict")
brief_lines.append(f"- Positive qty = buy, negative qty = sell")
brief_lines.append(f"- Position limits enforced server-side: violating = all orders cancelled")
brief_lines.append(f"- Three-phase pattern from top teams: **TAKE** (sweep mispriced) → **CLEAR** (reduce risk) → **MAKE** (passive quotes)")

with open(os.path.join(ai_export_dir, "strategy_brief.md"), "w") as f:
    f.write("\n".join(brief_lines))
print(f"[1/4] strategy_brief.md written ({len(brief_lines)} lines)")

# ─────────────────────────────────────────────────────────
# 2) MACHINE-READABLE PARAMETERS (JSON)
#    AI tools can parse this directly into code
# ─────────────────────────────────────────────────────────
params = {"round": 1, "generated": datetime.datetime.now().isoformat()}
params["products"] = {}
for p in PRODUCTS:
    df = FEAT[p]
    v = VERDICTS.get(p, "?")
    sr = SPREAD_RESULTS.get(p, {})
    acf = ACF_RESULTS.get(p, {})
    sig = SIGNAL_RESULTS.get(p, {})
    spk = SPIKE_RESULTS.get(p, {})
    pos = POS_RESULTS.get(p, {})
    _, avg_cv, drift_val, has_drift = classify_product_within_day(df)

    fv_stds = {m: float(df[m].std()) for m in ["simple_mid", "microprice", "wall_mid", "mm_mid"]}
    best_fv = min(fv_stds, key=fv_stds.get)

    signal_names_local = ["ret1", "z20", "micro_delta", "wall_delta", "mm_delta", "imbalance", "spread_chg"]
    signals_out = {}
    if sig:
        for s in signal_names_local:
            r = sig.get(s, {})
            signals_out[s] = {
                "corr_h1": round(r.get("corr", 0), 4),
                "corr_h5": round(r.get("corrs", {}).get(5, 0), 4),
                "corr_h20": round(r.get("corrs", {}).get(20, 0), 4),
                "hit_rate": round(r.get("hit", 0.5), 4),
                "use": abs(r.get("corr", 0)) > 0.02,
                "direction": "momentum" if r.get("corr", 0) > 0 else "contrarian"
            }
        best_sig = max(signal_names_local, key=lambda s: abs(sig.get(s, {}).get("corr", 0)))
    else:
        best_sig = None

    # Periodicities
    rets = df["ret"].dropna()
    long_lags = list(range(1, min(201, len(rets) // 5)))
    acfs_long = [rets.autocorr(lag=l) for l in long_lags]
    sig_thresh = 2 / np.sqrt(len(rets))
    acfs_arr = np.array(acfs_long)
    peaks = []
    for i in range(1, len(acfs_arr) - 1):
        if (abs(acfs_arr[i]) > sig_thresh and
            abs(acfs_arr[i]) > abs(acfs_arr[i-1]) and
            abs(acfs_arr[i]) > abs(acfs_arr[i+1])):
            peaks.append({"lag": long_lags[i], "acf": round(float(acfs_arr[i]), 4)})

    days = sorted(df["day"].unique())
    day_stats = []
    for day in days:
        dd = df[df["day"] == day]["mid_price"]
        day_stats.append({
            "day": int(day), "open": float(dd.iloc[0]), "close": float(dd.iloc[-1]),
            "low": float(dd.min()), "high": float(dd.max()),
            "mean": round(float(dd.mean()), 1), "std": round(float(dd.std()), 1)
        })

    params["products"][p] = {
        "position_limit": POSITION_LIMITS.get(p, 80),
        "classification": v,
        "within_day_cv": round(float(avg_cv), 6),
        "has_drift": bool(has_drift),
        "drift_per_day": round(float(drift_val), 0) if has_drift else 0,
        "recommended_strategy": strat_map.get(v, "MM"),
        "fair_value": {
            "best_estimator": best_fv,
            "stds": {k: round(v, 4) for k, v in fv_stds.items()},
            "mm_size_threshold": MM_SIZE_THRESHOLD,
        },
        "spread": {
            "mean": round(float(sr.get("mean", 0)), 2),
            "p25": round(float(sr.get("p25", 0)), 1),
            "p50": round(float(sr.get("p50", 0)), 1),
            "p75": round(float(sr.get("p75", 0)), 1),
            "take_edge": round(float(sr.get("take_edge", 0)), 2),
            "passive_edge": round(float(sr.get("passive_edge", 0)), 2),
        },
        "mean_reversion": {
            "lag1_acf": round(float(acf.get("lag1", 0)), 4),
            "verdict": "MEAN REVERTING" if acf.get("lag1", 0) < -0.05 else (
                "TRENDING" if acf.get("lag1", 0) > 0.05 else "NEUTRAL"),
        },
        "best_signal": best_sig,
        "signals": signals_out,
        "spikes": {
            "count": int(spk.get("n_spikes", 0)),
            "per_10k_ticks": round(float(spk.get("freq_per_10k", 0)), 1),
            "avg_magnitude": round(float(spk.get("avg_mag", 0)), 2),
            "reversion_tradeable": bool(spk.get("reversion", False)),
        },
        "periodicities": peaks[:8],
        "daily_stats": day_stats,
        "simulation": {
            "max_position": int(pos.get("max_pos", 0)),
            "pct_at_limit": round(float(pos.get("pct_at_limit", 0)), 1),
            "mtm_pnl": round(float(pos.get("final_pnl", 0)), 0),
        },
    }

class NumpyEncoder(json_mod.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.bool_,)): return bool(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

with open(os.path.join(ai_export_dir, "product_params.json"), "w") as f:
    json_mod.dump(params, f, indent=2, cls=NumpyEncoder)
print(f"[2/4] product_params.json written ({len(params['products'])} products)")

# ─────────────────────────────────────────────────────────
# 3) CURATED PLOTS — only those where visual > numbers
#    Token budget: ~5-8 images max (~10-15k tokens)
# ─────────────────────────────────────────────────────────
plots_dir = os.path.join(ai_export_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

# Selection criteria:
# HIGH VALUE (pattern recognition needed, can't convey as numbers):
#   - param_grid: 2D heatmap, best seen visually
#   - spike_reversion: shape of reversion curve matters
#   - periodicity: peak pattern in ACF plot
#   - signal_decay: which signals persist vs fade
# MEDIUM VALUE (useful backup):
#   - avg_intraday_path: daily shape pattern
# LOW VALUE (already captured as numbers in brief):
#   - price_history, spread_distribution, boxplots, raw overview charts
#   - dashboard: everything in it is already in the brief
#   - autocorrelation bars: single number captures it
#   - fair_value comparison: best FV already named

curated = []
for p in PRODUCTS:
    # Must-have: param sensitivity grid (visual pattern of stable vs fragile regions)
    curated.append(f"11A_param_grid_{p}.png")
    # Must-have: spike reversion curve (shape shows how quickly/reliably prices revert)
    curated.append(f"8B_spike_reversion_{p}.png")
    # Must-have: periodicity ACF (peak locations = hidden pattern, visual is clearest)
    curated.append(f"7C_periodicity_{p}.png")
    # Useful: signal decay shows which signals persist (multi-line plot)
    curated.append(f"5B_signal_decay_{p}.png")

# Cross-product (only 1 image, confirms independence)
if len(PRODUCTS) >= 2:
    curated.append("9B_correlation_matrix.png")

copied = 0
for fname in curated:
    src = os.path.join(PLOT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, plots_dir)
        copied += 1
print(f"[3/4] {copied} curated plots copied (from {len(os.listdir(PLOT_DIR))} total)")

# ─────────────────────────────────────────────────────────
# 4) DATA SAMPLE — compact format for AI to see structure
#    Full CSVs are wasteful; 50 rows per product is enough
# ─────────────────────────────────────────────────────────
data_dir = os.path.join(ai_export_dir, "data_samples")
os.makedirs(data_dir, exist_ok=True)

for p in PRODUCTS:
    df = FEAT[p]
    # First 50 rows with key columns only (not all 30+ columns)
    cols = ["day", "timestamp", "mid_price", "bid_price_1", "ask_price_1",
            "bid_volume_1", "ask_volume_1", "spread", "microprice", "wall_mid",
            "mm_mid", "imbalance", "ret", "z20"]
    sample = df[cols].head(50)
    sample.to_csv(os.path.join(data_dir, f"sample_{p}.csv"), index=False)

    # Trade sample
    if p in trade_dfs and len(trade_dfs[p]) > 0:
        trade_dfs[p].head(30).to_csv(os.path.join(data_dir, f"sample_trades_{p}.csv"), index=False)

print(f"[4/4] Data samples written (50 price rows + 30 trade rows per product)")

# ─────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"AI Strategy Context exported to:")
print(f"  {ai_export_dir}/")
print(f"{'='*60}")
print(f"Contents:")
print(f"  strategy_brief.md   — full analysis as structured text (~3-4k tokens)")
print(f"  product_params.json — machine-readable params for direct use in code")
print(f"  plots/              — {copied} curated images (~{copied * 1500 // 1000}k tokens)")
print(f"  data_samples/       — compact CSVs showing data format")
print(f"\nEstimated total context cost: ~{3 + copied * 1.5:.0f}k tokens")
print(f"vs all plots: ~{len(os.listdir(PLOT_DIR)) * 1.5:.0f}k tokens")
print(f"\nRecommended usage:")
print(f"  1. Always include strategy_brief.md + product_params.json")
print(f"  2. Add plots/ only when debugging specific behavior")
print(f"  3. Add data_samples/ when AI needs to understand data format")
